In [1]:
import pandas as pd

# Single CSV exploration: build_dataset.py and download.py development

In [2]:
# Load sin CSV
df = pd.read_csv("../data/raw/job-bank-open-data-all-job-postings-en-may2026.csv", encoding="utf-16", sep="\t", low_memory=False)

In [3]:

# Normalize column names: collapse multiple spaces, strip leading/trailing
df.columns = df.columns.str.strip().str.replace(r"\s+", " ", regex=True)

missing_city = df[df["City"].isna()]

In [4]:
print("Rows missing City:", len(missing_city))
print()
print("Their Province/Territory values:")
print(missing_city["Province/Territory"].value_counts())
print()
print("Their Economic Region values:")
print(missing_city["Economic Region"].value_counts())

Rows missing City: 465

Their Province/Territory values:
Province/Territory
Québec                       137
Ontario                      133
British Columbia              85
Alberta                       31
Nova Scotia                   31
Saskatchewan                  26
New Brunswick                  9
Manitoba                       7
Newfoundland and Labrador      2
Nunavut                        2
Prince Edward Island           2
Name: count, dtype: int64

Their Economic Region values:
Series([], Name: count, dtype: int64)


In [5]:
print("Also missing Economic Region:", missing_city["Economic Region"].isna().sum(), "/", len(missing_city))
print()
print("Postal code populated for these rows:", missing_city["Work Location Postal Code"].notna().sum(), "/", len(missing_city))
print()
print("Sample postal codes (first 3 chars = FSA):")
print(missing_city["Work Location Postal Code"].dropna().str[:3].value_counts().head(15))

Also missing Economic Region: 465 / 465

Postal code populated for these rows: 0 / 465

Sample postal codes (first 3 chars = FSA):
Series([], Name: count, dtype: int64)


# Final dataset revision

In [6]:
df1 = pd.read_csv("../data/processed/vancouver_tech_postings.csv", low_memory=False)

In [7]:
df1

,WIC Job Location Snapshot ID,Job Title,Original Job Title,NOC 2016 Code,NOC 2016 Code Name,NOC21 Code,NOC21 Code Name,External Indicator,First Posting Date,Vacancy Count,...,Condition Vision Care,Salary Condition Other Benefits,Commission PER,Commission Type,Hours Per,Hours Minimum,Hours Maximum,Work Hours,Work Hours From Time,Work Hours To Time
0,15538106.0,systems analyst,IT Systems Analyst (ESRI ArcGIS),2171.0,Information systems analysts and consultants,21222.0,Information systems specialists,1,2026-05-11,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN
1,15554305.0,business process analyst,NaN,2171.0,Information systems analysts and consultants,21221.0,Business systems specialists,0,2026-05-15,1,...,No,No,NaN,NaN,Week,40.0,40.0,Yes,09:00,17:00
2,15575608.0,information technology (IT) analyst,Government Relations Specialist,2171.0,Information systems analysts and consultants,21222.0,Information systems specialists,1,2026-05-23,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN
3,15568431.0,devops engineer,Mechatronics Engineer,2173.0,Software engineers and designers,21231.0,Software engineers and designers,1,2026-05-21,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN
4,15538267.0,software engineer,Senior Backend Engineer,2173.0,Software engineers and designers,21231.0,Software engineers and designers,1,2026-05-11,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4374,NaN,software engineer,"Senior Lead Software Engineer, Network Systems",2173.0,Software engineers and designers,21231.0,Software engineers and designers,1,2024-06-19,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN
4375,NaN,software developer,Senior Python Developer,2174.0,Computer programmers and interactive media dev...,21232.0,Software developers and programmers,1,2024-06-25,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN
4376,NaN,software engineer,Principal Software Engineer,2173.0,Software engineers and designers,21231.0,Software engineers and designers,1,2024-06-19,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN
4377,NaN,software developer,SAP iXp Intern - SAP Concur App Center Full St...,2174.0,Computer programmers and interactive media dev...,21232.0,Software developers and programmers,1,2024-06-19,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN


In [8]:
df1['City'].unique()

<StringArray>
[        'Burnaby',       'Coquitlam',        'Richmond',          'Surrey',
       'Vancouver', 'North Vancouver',  'West Vancouver',           'Delta',
 'New Westminster',  'Port Coquitlam',         'Langley',     'Maple Ridge',
      'White Rock',    'Pitt Meadows',      'Port Moody']
Length: 15, dtype: str

In [9]:
print(df1[["Salary Minimum", "Salary Maximum"]].notna().sum())
print(f"Total rows: {len(df1)}")

Salary Minimum    2928
Salary Maximum    2928
dtype: int64
Total rows: 4379


In [10]:
# How wide is Salary Minimum vs Maximum — some postings report ranges, others identical min/max
print((df1["Salary Maximum"] - df1["Salary Minimum"]).describe())

count      2928.000000
mean       5719.544580
std       13998.815911
min           0.000000
25%           0.000000
50%           0.000000
75%          47.690000
max      153050.000000
dtype: float64


In [11]:
# Sanity check for garbage values (e.g. $0, or suspiciously low/high salaries)
print(df1[["Salary Minimum", "Salary Maximum"]].describe())

       Salary Minimum  Salary Maximum
count     2928.000000     2928.000000
mean     33068.494413    38788.038992
std      47134.071662    55524.655597
min         10.000000       10.000000
25%         38.475000       40.500000
50%         53.850000       59.495000
75%      70000.000000    83500.000000
max     250000.000000   323050.000000


In [12]:
# Salary Per matters — hourly vs annual figures mixed together will wreck a model
print(df1["Salary Per"].value_counts())

Salary Per
Hour         1723
Year         1073
Month          79
Bi-weekly      52
Week            1
Name: count, dtype: int64


# Notes and Information: Raw dataset and final Metro Vancouver dataset

* Raw rows loaded: 1,783,070
* Note: 1,411 tech-NOC postings have no City (and no Economic Region / postal code fallback available) — excluded since they can't be confirmed as Metro Vancouver.
* Note: 597 rows matched a Metro Vancouver city name but were NOT in British Columbia — excluded. Examples:


| Number of data points | City       | Province/Territory   |
|-----------------------|------------|-----------------------|
| 366                    | Richmond   | Québec                |
| 4032                   | Richmond   | Ontario               |
| 4386                   | Delta      | Ontario               |
| 184770                 | White Rock | Nova Scotia           |
| 467368                 | Richmond   | Prince Edward Island  |

* Vancouver tech postings after filtering: 4,379